# Notebook dedicated to finding the inverse function of the teos10 implementation

**How to?**

First thought is the function of GSW which calculates the temperature and salinity from $\sigma_0$ and use a estimated temperature as a "spiciness" factor to estimate the right relationship between the temperature and the salinity. This will be the input for the reverse iterative Newton Raphson method - ie. the inverse of eta target. 

1st try will be derivated from the following GitHub site: 
https://github.com/TEOS-10/GSW-Matlab/blob/master/Toolbox/gsw_SA_CT_from_sigma0_spiciness0.m

1. Make a salinity estimate based on the density and the spiciness of the water
    - Calculate $\delta s$ by subtracting the real estimate of the spiciness
2. CT from $\rho$ - to use as a target in the Newton Raphson iterative process [$\rho$ = $\sigma_0 + 1000$, ie. the non-anomaly density]
    - Only need to store the conservative temperature
3. Newton Raphson iterative process to find $\delta S$ - then adjust the salinity estimate thereof

$$
SA = SA_{estimate} \quad + \delta s  \quad + \frac{D SA}{D s}
$$

4. Then find the final conservative temperature dependent on the final SA 

**OBS**
The function provided by GSW is limited to the salinity being $ SA < 0, 42 > SA $ and $ CT < 5, 40 > CT $



In [ ]:
def SA_polynomials(sigma0, spiciness0):
    """
    Function for calculating the absolute salinity polynomials - both standard SA and the differentiated polynomials 
    """
    SA_pol = {
        1 : 16.907145985921161,
        2 : 0.622223077618381,
        3 : 0.000223330499353,
        4 : 0.706643311920640,
        5 : 0.000454964160674,
        6 : 0.000312654210024,
        7 : 0.000103973707417,
        8 : 0.000113069609374,
        9 : 0.000026554807951,
        10: 0.000044377349558,}

    SA = SA_pol[1] + sigma0 * (SA_pol[2] + SA_pol[6] * spiciness0 + sigma0 * (SA_pol[3] + SA_pol[7] * spiciness0 + SA_pol[9] * sigma0)) + spiciness0 * (SA_pol[4] + spiciness0 * (SA_pol[5] + SA_pol[8] * sigma0 + SA_pol[10] * spiciness0))  
    SA_diff = SA_pol[4] + sigma0 * (SA_pol[6] + SA_pol[7] * sigma0) + spiciness0 * (2 * (SA_pol[5] + SA_pol[8] * sigma0) + 3 * SA_pol[10] * spiciness0)

    return SA, SA_diff

16.90714598592116